# OptiCrop - 02 Preprocessing and Model Building
This notebook handles preprocessing, stratified split, and compares four classification models to select the best predictor.

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("../data/Crop_recommendation.csv")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

FEATURE_COLUMNS = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET_COLUMN = "label"

## A. Load Data and Check Validity

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates().copy()
assert not df.isnull().any().any(), "Dataset contains null values!"
print("Dataset size after clean:", df.shape)

## B. Train-Test Split (80/20 Stratified Split)

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## C. Model Definitions and Training

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=42)),
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

trained_models = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    trained_models[name] = model
    results.append({"model": name, "accuracy": acc})
    print(f"{name} Accuracy: {acc:.4f}")

## D. Model Performance Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
display(results_df)

## E. Evaluate and Save Best Model

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = trained_models[best_name]
final_pred = best_model.predict(X_test)

print(f"Selected Best Model: {best_name}\n")
print(classification_report(y_test, final_pred))

# Save confusion matrix
labels = sorted(y.unique())
fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(y_test, final_pred, labels=labels, ax=ax, cmap="YlGn", colorbar=False)
plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.savefig("../screenshots/graphs/confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

# Dump model files
joblib.dump(best_model, MODEL_DIR / "crop_model.pkl")
joblib.dump(FEATURE_COLUMNS, MODEL_DIR / "feature_columns.pkl")
joblib.dump(best_name, MODEL_DIR / "model_name.pkl")
print("Model pipelines saved successfully in models/")